# SASRec baseline on MovieLens-1M

**SASRec** (Kang & McAuley, 2018, "Self-Attentive Sequential Recommendation") is the standard sequential
recommender: a small transformer that reads a user's rating history (movie IDs only - no titles, plots or genres)
and predicts the next movie (`genrec.models.sasrec`). It is the usual yardstick for models like the fine-tuned Qwen.

Same split and test as every other model: train on `items[:-2]`, validate on `items[-2]`, test on `items[-1]`; the
same 1,000 test users; full ranking over the catalogue, already-rated movies skipped; hit@1, HR@10, NDCG@10.

Two variants: **SASRec-200** (the standard setup, up to 200 past ratings) and **SASRec-20** (only the last 20, the
same information the Qwen prompt gets). Trains in ~15 minutes on a laptop; the models are saved to `artifacts/`
for the serving endpoint.

Note on numbers in papers: the SASRec paper reports Hit@10 = 0.82 on ML-1M because it ranks the true movie among
only 100 sampled negatives; here every movie in the catalogue is a candidate (full ranking), which is much harder
and the more reliable protocol (Krichene & Rendle, 2020).

In [1]:
import pandas as pd
import torch

from genrec.data import Dataset, history, sample_test_users, target
from genrec.models import sasrec
from genrec.paths import ARTIFACTS_DIR, ML1M_DIR, RESULTS_DIR

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
VARIANTS = {"SASRec-200": sasrec.SASRecConfig(max_len=200), "SASRec-20": sasrec.SASRecConfig(max_len=20)}
print("device:", DEVICE)

device: mps


## 1. Data

In [2]:
ds = Dataset.load(ML1M_DIR)
seqs = ds.encoded()
test_users = sample_test_users(ds.users)
title = dict(zip(range(1, ds.index.n_cols), ds.movies.set_index("MovieID").loc[ds.index.movie_ids, "Title"]))
lengths = pd.Series([len(s) for s in seqs.values()])
print(f"{len(seqs)} users, {ds.index.n_cols - 1} movies, {len(ds.ratings)} ratings; "
      f"ratings per user: median {lengths.median():.0f}, max {lengths.max()}")

6040 users, 3706 movies, 1000209 ratings; ratings per user: median 96, max 2314


## 2. Train both variants

Every epoch: a random window of each user's training history, every position predicted (full softmax). The epoch
with the best validation NDCG@10 is kept.

In [3]:
models = {}
for name, cfg in VARIANTS.items():
    print(f"--- {name}")
    models[name], _ = sasrec.train(seqs, ds.index.n_cols - 1, cfg, DEVICE, log=lambda m, n=name: print(f"{n} {m}"))
    sasrec.save(models[name], ds.index.movie_ids, ARTIFACTS_DIR / name.lower())
print("saved to", ARTIFACTS_DIR)

--- SASRec-200


SASRec-200 epoch   1  loss   366.0  val HR@10 0.041  NDCG@10 0.019  (best 0.019 @ 1)  6s


SASRec-200 epoch  10  loss   292.8  val HR@10 0.210  NDCG@10 0.116  (best 0.116 @ 10)  53s


SASRec-200 epoch  20  loss   275.7  val HR@10 0.278  NDCG@10 0.152  (best 0.152 @ 20)  107s


SASRec-200 epoch  30  loss   270.1  val HR@10 0.293  NDCG@10 0.160  (best 0.161 @ 29)  158s


...

SASRec-20 epoch 110  loss   261.2  val HR@10 0.297  NDCG@10 0.167  (best 0.169 @ 106)  66s


SASRec-20 epoch 120  loss   260.5  val HR@10 0.302  NDCG@10 0.168  (best 0.169 @ 117)  71s


SASRec-20 epoch 130  loss   259.7  val HR@10 0.297  NDCG@10 0.167  (best 0.171 @ 124)  77s


SASRec-20 epoch 140  loss   259.8  val HR@10 0.299  NDCG@10 0.169  (best 0.171 @ 124)  83s


SASRec-20 stopped at epoch 144, best validation NDCG@10 0.171 at epoch 124, 85s, 0.29M params
saved to artifacts


## 3. Test-set comparison

In [4]:
def test_metrics(model, us):
    return sasrec.evaluate(model, [history(seqs[u], "test") for u in us], [target(seqs[u], "test") for u in us], DEVICE)


results = {name: test_metrics(m, test_users) for name, m in models.items()}
table = pd.DataFrame(results).T[["hit@1", "HR@10", "NDCG@10"]]
RESULTS_DIR.mkdir(exist_ok=True)
table.to_csv(RESULTS_DIR / "sasrec_test.csv")   # picked up by baselines.ipynb
print(f"test: last rating of {len(test_users)} users")
table.style.format("{:.3f}")

test: last rating of 1000 users


,hit@1,HR@10,NDCG@10
SASRec-200,0.080,0.294,0.171
SASRec-20,0.066,0.263,0.152


### All 6,040 users

In [5]:
pd.DataFrame({name: test_metrics(m, ds.users) for name, m in models.items()}).T.style.format("{:.3f}")

,hit@1,HR@10,NDCG@10
SASRec-200,0.076,0.306,0.176
SASRec-20,0.066,0.284,0.161


## 4. Example recommendations (SASRec-200)

In [6]:
for u in test_users[:5]:
    hist, tgt = history(seqs[u], "test"), target(seqs[u], "test")
    top = sasrec.recommend(models["SASRec-200"], hist, DEVICE)
    verdict = f"HIT at rank {top.index(tgt) + 1}" if tgt in top else "miss"
    print(f"user {u}  last rated: {'; '.join(title[m] for m in hist[-5:])}")
    print(f"   real next: {title[tgt]}   {verdict}")
    print(f"   top 10:    {'; '.join(title[m] for m in top)}\n")

user 3156  last rated: Gone in 60 Seconds (2000); Almost Famous (2000); Best in Show (2000); Tao of Steve, The (2000); Meet the Parents (2000)
   real next: Space Cowboys (2000)   HIT at rank 10
   top 10:    Nurse Betty (2000); Cell, The (2000); Contender, The (2000); Remember the Titans (2000); Bring It On (2000); Wonder Boys (2000); Original Kings of Comedy, The (2000); Duets (2000); Art of War, The (2000); Space Cowboys (2000)

user 3446  last rated: Frighteners, The (1996); Fright Night (1985); Phantasm IV: Oblivion (1998); Nightmare on Elm Street, A (1984); Blob, The (1958)
   real next: Near Dark (1987)   miss
   top 10:    Gremlins (1984); Little Shop of Horrors (1986); Bram Stoker's Dracula (1992); Phantasm (1979); Omen, The (1976); Blair Witch Project, The (1999); Howling, The (1980); Hunger, The (1983); Interview with the Vampire (1994); American Werewolf in London, An (1981)

user 332  last rated: Annie Hall (1977); Home for the Holidays (1995); Usual Suspects, The (1995); 